# Parse Minnie Clustering — Step-by-Step Workflow

This notebook walks through **how this repository uses LinkML schemas and auto-generated Pydantic models** to build and write connectivity/clustering data. It is a trimmed, commented version of `parse_minnie_clustering.ipynb`.

**Pipeline overview:**
1. **Schemas** are defined in LinkML (YAML under `schemas/`).
2. **Pydantic models** are generated from those schemas (e.g. via `scripts/generate_models.sh`).
3. You **instantiate** those models with your data (e.g. `DataSet`, `DataItem`, `Cluster`).
4. **Arrow utilities** turn model instances into PyArrow tables: `build_arrow_schema()`, `models_to_table()`, `attach_linkml_metadata()`.
5. Tables are written to **Delta Lake** with `write_deltalake()` for storage and querying.


In [ ]:
# Imports: package models (from LinkML) and Arrow/Delta helpers
import sys
from pathlib import Path

import pandas as pd
import pyarrow as pa
from deltalake import write_deltalake

# Add repo root so we can import the generated package (run from repo root or code/)
_repo = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(_repo / "src"))

from connects_common_connectivity.arrow_utils import (
    build_arrow_schema,
    models_to_table,
    attach_linkml_metadata,
    build_cell_feature_matrix_schema,
)
from connects_common_connectivity.models import (
    DataSet,
    DataItem,
    DataItemDataSetAssociation,
    Unit,
    CellFeatureDefinition,
    CellFeatureSet,
    Cluster,
    ClusterMembership,
    CellCellConnectivityLong,
    SynapticMeasurementType,
)

---

## 1. Where do the models come from?

The classes (`DataSet`, `DataItem`, `Cluster`, etc.) are **generated from LinkML YAML schemas** in `schemas/`. The script `scripts/generate_models.sh` (typically using `linkml-codegen` or similar) produces Python Pydantic models in `src/connects_common_connectivity/models.py`. When the YAML changes, you re-run the generator to refresh the models.

## 2. Creating a DataSet (domain object)

You create **instances of the generated Pydantic models** like any other Python class. Here we define a single dataset that will group our cells/items.

In [ ]:
ds = DataSet(
    id="minnie65_v1412_csm_cluster",
    name="Minnie65 v1412 CSM Dendrite Ultrastructure Collection",
    publication="none",
    modality="ELECTRON_MICROSCOPY",
    project_id="minnie65",
)
ds

## 3. From Pydantic models to Arrow tables

Three steps are used everywhere in this workflow:

1. **`build_arrow_schema(SomeModel)`** — Builds a PyArrow schema from the Pydantic model’s field types (so column types are consistent).
2. **`models_to_table(instances, schema)`** — Converts a list of model instances into a single PyArrow table (no JSON round-trip).
3. **`attach_linkml_metadata(table, linkml_class="...")`** — Attaches schema/class metadata to the table for provenance (e.g. which LinkML class and version).

In [ ]:
schema = build_arrow_schema(DataSet)
table = models_to_table([ds], schema)
table = attach_linkml_metadata(table, linkml_class="DataSet")
table

## 4. Writing to Delta Lake

Tables are written with **`write_deltalake()`**. Using `partition_by` (e.g. `project_id`) keeps the layout consistent and makes filtering by project efficient. Use a path where you want the Delta table to live (e.g. under `../results/`).

In [ ]:
# Uncomment and set PATH to a directory where you want the Delta table (e.g. ../results/dataset/)
# PATH = "../results/dataset/"
# write_deltalake(PATH, table, mode="append", partition_by=["project_id"])

## 5. DataItems and DataItem–DataSet associations

- **DataItem**: Represents a single “thing” in the project (e.g. a nucleus/cell), with `id`, `name`, `project_id`.
- **DataItemDataSetAssociation**: Links each DataItem to a DataSet (many-to-many). So you can have multiple datasets and assign each cell to one or more of them.

In the original notebook, DataItems are built by iterating over a CAVE/materialization table; here we use a small in-memory list for illustration.

In [ ]:
# Example: a few DataItems (in the full pipeline these come from e.g. nuc_df)
data_items = [
    DataItem(id="100", name="864691136090135607", project_id="minnie65"),
    DataItem(id="101", name="864691136662334942", project_id="minnie65"),
    DataItem(id="102", name="864691136145011252", project_id="minnie65"),
]

schema = build_arrow_schema(DataItem)
table = models_to_table(data_items, schema)
table = attach_linkml_metadata(table, linkml_class="DataItem")
# write_deltalake("../results/dataitem/", table, mode="append", partition_by=["project_id"])
table

In [ ]:
# Link each DataItem to the DataSet we created earlier
data_item_associations = [
    DataItemDataSetAssociation(dataitem_id=item.id, dataset_id=ds.id, project_id="minnie65")
    for item in data_items
]

schema = build_arrow_schema(DataItemDataSetAssociation)
table = models_to_table(data_item_associations, schema)
table = attach_linkml_metadata(table, linkml_class="DataItemDataSetAssociation")
table

## 6. Querying Delta tables (Polars)

Once data is written to Delta, you **read it with Polars** (or Pandas) and join tables as needed. Typical pattern: filter associations by `project_id` and `dataset_id`, then join to the DataItem table on `dataitem_id` = `id` to get the full item metadata for that dataset.

In [ ]:
import polars as pl

# If you wrote Delta tables to ../results/, you can run:
# assoc_df = pl.read_delta("../results/dataitem_dataset_association/")
# items_df = pl.read_delta("../results/dataitem/")
# result = (
#     assoc_df
#     .filter(pl.col("project_id") == "minnie65")
#     .filter(pl.col("dataset_id") == ds.id)
#     .join(items_df, left_on="dataitem_id", right_on="id", how="inner")
# )
# print(result)

# Here we simulate the same join on the in-memory tables we just built
assoc_df = pl.from_arrow(models_to_table(data_item_associations, build_arrow_schema(DataItemDataSetAssociation)))
items_df = pl.from_arrow(models_to_table(data_items, build_arrow_schema(DataItem)))
result = assoc_df.join(items_df, left_on="dataitem_id", right_on="id", how="inner")
result

## 7. Cell features: definitions and feature sets

- **CellFeatureDefinition**: Describes one named feature (id, description, unit, data_type, optional range_min/range_max). The original notebook loads these from a CSV (e.g. `minnie_cell_features.csv`).
- **CellFeatureSet**: Groups a list of feature definition IDs and describes the set (id, description, feature_definition_ids, extraction_method). One feature set can be used for many cells.

In [ ]:
# Example: a couple of feature definitions and one feature set
fd1 = CellFeatureDefinition(
    id="nucleus_volume_um",
    description="Nucleus volume",
    unit="MICRONS_CUBED",
    data_type="<f4",
    range_min=0.0,
    range_max=None,
)
fd2 = CellFeatureDefinition(
    id="nucleus_area_um",
    description="Nucleus surface area",
    unit="MICRONS_SQUARE",
    data_type="<f4",
)
fds = [fd1, fd2]

cfs = CellFeatureSet(
    id="csm_cluster_features",
    description="Example cell features for clustering.",
    feature_definition_ids=[fd.id for fd in fds],
    extraction_method="Aggregated from skeleton feature extraction.",
)

schema_fd = build_arrow_schema(CellFeatureDefinition)
table_fd = models_to_table(fds, schema_fd)
table_fd = attach_linkml_metadata(table_fd, linkml_class="CellFeatureDefinition")

schema_cfs = build_arrow_schema(CellFeatureSet)
table_cfs = models_to_table([cfs], schema_cfs)
table_cfs = attach_linkml_metadata(table_cfs, linkml_class="CellFeatureSet")
table_cfs

## 8. Cell feature matrix (per-cell feature values)

Feature **definitions** describe what each column means; the **feature matrix** is the table of actual values: one row per cell (e.g. `id`), one column per feature, plus `project_id` and `feature_set_id`. **`build_cell_feature_matrix_schema(feature_set, list_of_definitions, cell_index_column="id")`** returns a PyArrow schema that matches this layout. You then build a DataFrame with the same columns and cast types to match the definitions (e.g. float32 for `<f4`), and convert to a PyArrow table with that schema before writing to Delta.

In [ ]:
# Build a schema for a table: id (cell), feature columns, project_id, feature_set_id
schema = build_cell_feature_matrix_schema(cfs, fds, cell_index_column="id")

# Example: tiny feature matrix (in the full pipeline this comes from a large DataFrame)
import pandas as pd
df = pd.DataFrame({
    "id": ["100", "101", "102"],
    "nucleus_volume_um": [344.1, 254.5, 338.0],
    "nucleus_area_um": [269.3, 250.4, 298.6],
    "project_id": "minnie65",
    "feature_set_id": "csm_cluster_features",
})
for cfd in fds:
    col = cfd.id
    if col in df.columns and cfd.data_type and cfd.data_type[1] == "f":
        df[col] = df[col].astype("float32")

table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)
table

## 9. Clusters and ClusterMembership

- **Cluster**: A node in a taxonomy (e.g. neuron → glutamatergic / gabaergic → L4IT, PTC, …). Fields include `id`, `parent`, `children`, `level`, `hex_color`, `heirachy_category`, `project_id`.
- **ClusterMembership**: Assigns each “item” (e.g. cell id) to a cluster, optionally with `probability`. The original notebook builds clusters from CAVE’s `cell_type_multifeature_v1` and then one membership row per (cell, cluster) at each level (neuron, class, subtype).

In [ ]:
# Minimal hierarchy: root and two children (in the full notebook this is built from cell_type_multifeature_v1)
nrn_cluster = Cluster(
    id="neuron",
    children=["glutamatergic", "gabaergic"],
    level=0,
    hex_color="#000000",
    heirachy_category="major_class",
    project_id="minnie65",
)
exc_cluster = Cluster(
    id="glutamatergic",
    parent=nrn_cluster.id,
    children=["L4IT", "L6CT"],
    hex_color="#FF0000",
    heirachy_category="class",
    level=1,
    project_id="minnie65",
)
inh_cluster = Cluster(
    id="gabaergic",
    parent=nrn_cluster.id,
    children=["PTC", "ITC"],
    hex_color="#0000FF",
    heirachy_category="class",
    level=1,
    project_id="minnie65",
)
clusters = [nrn_cluster, exc_cluster, inh_cluster]

schema = build_arrow_schema(Cluster)
table = models_to_table(clusters, schema)
table = attach_linkml_metadata(table, linkml_class="Cluster")
table

In [ ]:
# ClusterMembership: which item belongs to which cluster (with optional probability)
cms = [
    ClusterMembership(item="100", cluster="neuron", probability=1.0, project_id="minnie65"),
    ClusterMembership(item="100", cluster="glutamatergic", probability=1.0, project_id="minnie65"),
    ClusterMembership(item="100", cluster="L4IT", project_id="minnie65"),
    ClusterMembership(item="101", cluster="neuron", probability=1.0, project_id="minnie65"),
    ClusterMembership(item="101", cluster="gabaergic", probability=1.0, project_id="minnie65"),
]

schema = build_arrow_schema(ClusterMembership)
table = models_to_table(cms, schema)
table = attach_linkml_metadata(table, linkml_class="ClusterMembership")
table

## 10. Cell–cell connectivity (long format)

**CellCellConnectivityLong** stores one row per (presynaptic_cell, postsynaptic_cell, measurement_type): e.g. synapse count or sum anatomical size. Each row has `id`, `presynaptic_cell`, `postsynaptic_cell`, `measurement_type` (enum, e.g. `SYNAPSE_COUNT`, `SUM_ANATOMICAL_SIZE`), `value`, `unit`, `project_id`. The original notebook reads a connectivity parquet, filters by proofread axons, then builds two Delta tables—one for synapse count and one for sum size—partitioned by `project_id` and `measurement_type`.

In [ ]:
# Example: a few connectivity rows (synapse count)
cccls = [
    CellCellConnectivityLong(
        id="1",
        presynaptic_cell="337175",
        postsynaptic_cell="304043",
        measurement_type=SynapticMeasurementType.SYNAPSE_COUNT,
        value=1,
        unit=Unit.COUNT,
        project_id="minnie65",
    ),
    CellCellConnectivityLong(
        id="2",
        presynaptic_cell="330167",
        postsynaptic_cell="339142",
        measurement_type=SynapticMeasurementType.SYNAPSE_COUNT,
        value=1,
        unit=Unit.COUNT,
        project_id="minnie65",
    ),
]

schema = build_arrow_schema(CellCellConnectivityLong)
table = models_to_table(cccls, schema)
table = attach_linkml_metadata(table, linkml_class="CellCellConnectivityLong")
# write_deltalake("../results/cellcellconnectivitylong/", table, mode="append",
#                 partition_by=["project_id", "measurement_type"])
table

---

## Summary

| Step | What it does |
|------|----------------|
| **LinkML YAML** | Defines classes and slots in `schemas/`. |
| **generate_models** | Produces Pydantic models in `models.py`. |
| **Your code** | Instantiates models (DataSet, DataItem, Cluster, …) from your data sources. |
| **build_arrow_schema(Model)** | Gets a PyArrow schema for that model. |
| **models_to_table(instances, schema)** | Converts model instances to one PyArrow table. |
| **attach_linkml_metadata(table, linkml_class=...)** | Adds schema/version metadata. |
| **write_deltalake(path, table, partition_by=...)** | Writes the table to Delta Lake. |
| **Polars/Pandas** | Read Delta and join/filter for downstream analysis. |

The full `parse_minnie_clustering.ipynb` does the same pattern at scale: CAVE/materialization tables and parquet inputs → model instances → Arrow → Delta; plus coordinate transforms, UMAP, and proofread-only datasets. This notebook keeps only the **reusable workflow** and explains each part.